# Foundation Models Quickstart: CausalPFN

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/layer6ai-labs/causalfm-survey/blob/main/notebooks/Foundation_models_quickstart.ipynb)

The minimal code to run a working causal foundation model. This notebook
runs CausalPFN on one simulated dataset.

**Run this on Colab** (badge above) for zero setup. everything installs
automatically. Running locally instead needs `uv sync` first (see
`CLAUDE.md`).

Want to compare all three foundation models (CausalPFN, Do-PFN, CausalFM)
side-by-side? See
[`Foundation_models_sandbox.ipynb`](Foundation_models_sandbox.ipynb).

## Data generation process



In [1]:
#@title Data generation { display-mode: "form" }
import numpy as np
from sklearn.model_selection import train_test_split

# Discount email's effect on next-month spend. Treatment is confounded --
# more likely for loyal, high-spend customers -- so a naive comparison is biased.
SEED = 42
rng = np.random.default_rng(SEED)
n = 1500

recency, monetary, age = rng.normal(0, 1, (3, n)).astype(np.float32)
X = np.column_stack([recency, monetary, age])

propensity = 1 / (1 + np.exp(-(0.8 * monetary - 0.6 * recency)))
T = rng.binomial(1, propensity).astype(np.float32)

true_cate = (2.0 + 1.5 * recency - 0.75 * age).astype(np.float32)  # unobservable outside a simulation
Y0 = (5.0 + 2.0 * monetary - 0.5 * age + rng.normal(0, 1, n)).astype(np.float32)
Y = np.where(T == 1, Y0 + true_cate, Y0).astype(np.float32)

# ctx = context (conditioned on), qry = points to predict.
X_ctx, X_qry, T_ctx, T_qry, Y_ctx, Y_qry, _, tau_qry = train_test_split(
    X, T, Y, true_cate, test_size=0.3, random_state=SEED
)
true_ate = float(true_cate.mean())
print(f"True ATE (known only in this simulation): {true_ate:.3f}")

True ATE (known only in this simulation): 1.967


## Run CausalPFN

In [2]:
import importlib.util, platform, torch

if importlib.util.find_spec("causalpfn") is None:
    %pip install -q causalpfn  # also downloads pretrained weights on first use

device = "cuda" if torch.cuda.is_available() else "cpu"
# CausalPFN crashes on Apple Silicon macOS.
skip_apple_silicon = device != "cuda" and platform.system() == "Darwin" and platform.machine() == "arm64"

In [ ]:
if skip_apple_silicon:
    print("Skipping: CausalPFN segfaults on Apple Silicon macOS -- run this on Colab instead.")
else:
    from causalpfn import CATEEstimator, ATEEstimator

    # fit() doesn't train. it loads (X_ctx, T_ctx, Y_ctx) as context into the
    # frozen, pretrained transformer. estimate_cate/estimate_ate then run one
    # forward pass over the query set: no gradient steps, no hyperparameters.
    cate_estimator = CATEEstimator(device=device, verbose=False)
    cate_estimator.fit(X_ctx, T_ctx, Y_ctx)
    tau_hat = np.asarray(cate_estimator.estimate_cate(X_qry)).reshape(-1)

    ate_estimator = ATEEstimator(device=device, verbose=False)
    ate_estimator.fit(X_ctx, T_ctx, Y_ctx)
    ate_hat = float(np.asarray(ate_estimator.estimate_ate()).reshape(-1)[0])

    pehe = float(np.sqrt(np.mean((tau_hat - tau_qry) ** 2)))

Skipping: CausalPFN segfaults on Apple Silicon macOS -- run this on Colab instead.


In [4]:
if not skip_apple_silicon:
    print(f"ATE_hat={ate_hat:.3f}  True_ATE={true_ate:.3f}  PEHE={pehe:.3f}")

## Reference output

When `SEED = 42`, your results should match these numbers (verified Colab GPU run):

```
True ATE (known only in this simulation): 1.967
ATE_hat=1.911  True_ATE=1.967  PEHE=0.237
```